In [1]:
import os
# os.environ['ATTN_BACKEND'] = 'xformers'   # Can be 'flash-attn' or 'xformers', default is 'flash-attn'
os.environ['SPCONV_ALGO'] = 'native'        # Can be 'native' or 'auto', default is 'auto'.
                                            # 'auto' is faster but will do benchmarking at the beginning.
                                            # Recommended to set to 'native' if run only once.

os.environ['TORCH_CUDA_ARCH_LIST']='8.9'
# Suppress FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
from pathlib import Path

import numpy as np
import imageio
from PIL import Image
# from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.pipelines import TrellisImageToSparse3DPipeline
from trellis.utils import render_utils

[SPARSE] Backend: spconv, Attention: flash_attn
Warp 1.6.0 initialized:
   CUDA Toolkit 12.8, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3080 Laptop GPU" (16 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/atlas2/.cache/warp/1.6.0


In [ ]:
# Load a pipeline from a model folder or a Hugging Face model hub.
pipeline = TrellisImageToSparse3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipeline.cuda()

[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: flash_attn


Using cache found in /home/atlas2/.cache/torch/hub/facebookresearch_dinov2_main
/home/atlas2/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/atlas2/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/atlas2/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


In [4]:
images = list(Path("assets/example_multi_image").glob("*.png"))
images.sort()
sample_images = [Image.open(img) for img in images[:3]]

In [5]:
# config parameters 
sampling_steps = 12

In [6]:
# Run the pipeline
outputs = pipeline.run_multi_image(
    sample_images,
    seed=1,
    # Optional parameters
    sparse_structure_sampler_params={
        "steps": sampling_steps,
        "cfg_strength": 7.5,
    },
    slat_sampler_params={
        "steps": sampling_steps,
        "cfg_strength": 3,
    },
    formats='gaussian'
)

2025-02-18 12:55:52.464 | INFO     | trellis.pipelines.trellis_image_to_3d:inject_sampler_multi_image:337 - <bound method TrellisImageTo3DPipeline.inject_sampler_multi_image.<locals>._new_inference_model of <trellis.pipelines.samplers.flow_euler.FlowEulerGuidanceIntervalSampler object at 0x75c18f2af280>>
Sampling: 100%|██████████| 12/12 [00:04<00:00,  2.84it/s]
2025-02-18 12:55:56.938 | INFO     | trellis.pipelines.trellis_image_to_3d:inject_sampler_multi_image:337 - <bound method TrellisImageTo3DPipeline.inject_sampler_multi_image.<locals>._new_inference_model of <trellis.pipelines.samplers.flow_euler.FlowEulerGuidanceIntervalSampler object at 0x75bfe3f63070>>
Sampling: 100%|██████████| 12/12 [00:01<00:00,  6.81it/s]


In [11]:
outputs[1]['gaussian']

In [26]:
# saving the generated gaussian 3D model as ply file
outputs['gaussian'][0].save_ply(f'reconstructed_examples/{images[0].stem}_steps_{sampling_steps}.ply')

data loading pipeline

In [24]:
from pathlib import Path 
from PIL import Image
data_path = Path("/home/atlas2/work/data/rendered_images/bad_miner_dataset/")

In [ ]:
data_path = Path("/home/atlas2/work/data/rendered_images/bad_miner_dataset/")
for _dir in data_path.iterdir():
    dir_name = _dir.stem 
    img_list = list(_dir.glob("*.png"))
    img_list.sort()
    total_models = len(img_list) // 16
    for model_id in range(total_models):
        model_name = img_list[0 + model_id*16].stem.split(".")[0]
        sample_images = [Image.open(img_list[0 + model_id*16]), Image.open(img_list[6 + model_id*16]), Image.open(img_list[8 + model_id*16])]
        # print(img_list[0 + model*16], img_list[6 + model*16], img_list[8 + model*16])
    # print(_dir.stem, len(list(_dir.glob("*.png"))))

/home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/bulky_orange_robot_with_tank-like_0.png /home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/bulky_orange_robot_with_tank-like_14.png /home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/bulky_orange_robot_with_tank-like_2.png
/home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/dark_brown_hockey_stick_sharp_0.png /home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/dark_brown_hockey_stick_sharp_14.png /home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/dark_brown_hockey_stick_sharp_2.png
/home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/dark_green_moss_ogre_0.png /home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/dark_green_moss_ogre_14.png /home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/dark_green_moss_ogre_2.png
/home/atlas2/work/data/rendered_images/bad_miner_dataset/bad_miner/golden_shining_pow

In [38]:
_dir.stem

'bad_miner_trellis'

In [37]:
model_name

['tiny_green_metal_fork', 'trellis', '0', '74_0']